# RakshakAI v2 — QLoRA Fine-Tune

Trains **Qwen2.5-Coder-7B-Instruct** on curated CWE vulnerability data.

**Goal**: Create a security-specialized LLM that classifies and fixes code vulnerabilities.

**Method**: QLoRA (4-bit NF4, rank 16, ~40M trainable params)

**Dataset**: 80K curated CWE analysis records (248 CWE classes, 20 languages)


## Step 1: Install Dependencies

In [ ]:
!pip install -qU transformers peft accelerate bitsandbytes trl datasets huggingface_hub wandb

## Step 2: Imports & Config

In [ ]:
import os, json, gc, torch
import transformers
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from huggingface_hub import login as hf_login
from collections import Counter

In [ ]:
# Model
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Dataset — change to your HuggingFace dataset path
HF_DATASET = "Muneerali199/rakshak-cwe-v3-data"  # Change this!

OUTPUT_DIR = "/kaggle/working/rakshak-cwe-v3"
HF_REPO = "Muneerali199/rakshak-cwe-v3"  # Where to push the trained adapter

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
MAX_STEPS = 3000
LEARNING_RATE = 2e-4
SAVE_STEPS = 500
EVAL_STEPS = 500
LOGGING_STEPS = 25
WARMUP_STEPS = 100
MAX_SEQ_LENGTH = 2048

# Get HF token from Kaggle Secrets
HF_TOKEN = os.environ.get("HF_TOKEN", "")
print(f"HF_TOKEN set: {bool(HF_TOKEN)}")

## Step 3: Load Model (4-bit QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
print("Model loaded")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")

## Step 4: Apply LoRA Adapters

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Step 5: Load Dataset

In [ ]:
print(f"Loading dataset from {HF_DATASET}...")
dataset = load_dataset(HF_DATASET, split="train")
dataset = dataset.shuffle(seed=42)

split = dataset.train_test_split(test_size=0.02, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Check CWE distribution
cwes = Counter()
for i in range(min(len(train_dataset), 10000)):
    cwe = train_dataset[i].get("_meta", {}).get("cwe", "UNKNOWN")
    cwes[cwe] += 1
print("\nTop 10 CWEs (first 10K samples):")
for cwe, count in cwes.most_common(10):
    print(f"  {cwe}: {count}")

## Step 6: Format Data

In [ ]:
def format_chat(example):
    """Format messages using Qwen's chat template."""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

# Test on one sample
sample = format_chat(train_dataset[0])
print(sample["text"][:500])

## Step 7: Train

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    gradient_checkpointing=True,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    evaluation_strategy="steps",
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
    ddp_find_unused_parameters=False,
    remove_unused_columns=True,
    dataloader_num_workers=2,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_chat,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field=None,
    packing=False,
)

In [ ]:
trainer.train()

## Step 8: Save & Push to Hub

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

if HF_TOKEN:
    hf_login(HF_TOKEN)
    print(f"Pushing adapter to {HF_REPO}...")
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")
else:
    print("No HF_TOKEN. Download adapter from Kaggle output.")
    print(f"Files: {os.listdir(OUTPUT_DIR)}")

## Step 9: Quick Evaluation

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model, tokenizer=tokenizer,
    device_map="auto", max_new_tokens=256,
    do_sample=False, temperature=0.1,
)

test_cases = [
    ("C: gets() overflow", "c", 'char buf[10]; gets(buf);'),
    ("Python: SQL injection", "python", 'cursor.execute("SELECT * FROM users WHERE id = " + user_id)'),
    ("C: use-after-free", "c", 'free(ptr);\nprintf("%s", ptr->data);'),
]

for label, lang, code in test_cases:
    prompt = tokenizer.apply_chat_template([
        {"role": "system", "content": "You are a security expert. Analyze code for vulnerabilities."},
        {"role": "user", "content": f""""{lang}\n{code}\n""""},
    ], tokenize=False, add_generation_prompt=True)
    
    result = pipe(prompt)[0]["generated_text"]
    output = result[len(prompt):500]
    print(f"\n=== {label} ===")
    print(output)

## Step 10: Cleanup

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print("Done!")